# Model Evaluation with DeepEval — Output Quality (Simple Workflow)

A minimal **model quality evaluation** using [DeepEval](https://github.com/confident-ai/deepeval), complementing this repo's performance suites.

> **Where this fits:** everything else in this repo measures *performance/UX quality* (TTFT, ITL, goodput) via AIPerf — explicitly **not** output correctness (see `docs/metrics/model-selection.md`). This notebook covers that missing axis: *does the model's output actually answer the prompt correctly?* Model selection needs both — a fast model that answers wrong is not a candidate.

The workflow is the standard eval loop, kept deliberately small:

1. **Golden dataset** — curated `(input, expected_output)` pairs
2. **Generate** — send each input to the model under test (any OpenAI-compatible endpoint: NIM, vLLM, TGI, ...)
3. **Test cases** — bundle input / actual / expected into DeepEval `LLMTestCase`s
4. **Metrics** — score each case with LLM-as-judge metrics (GEval *Correctness*, *Answer Relevancy*)
5. **Results** — per-case scores + reasons, aggregate pass rate, saved artifact

## Table of contents

1. [Requirements & configuration](#1-requirements--configuration)
2. [Golden dataset](#2-golden-dataset)
3. [Generate outputs from the model under test](#3-generate-outputs-from-the-model-under-test)
4. [Build test cases & metrics](#4-build-test-cases--metrics)
5. [Run the evaluation](#5-run-the-evaluation)
6. [Results](#6-results)

## 1. Requirements & configuration

- `pip install -r notebooks/requirements.txt` (includes `deepeval` and the `openai` client).
- A reachable **OpenAI-compatible endpoint** serving the model under test — same assumption as every scenario in this repo.
- A **judge model** for the LLM-as-judge metrics. Pick one of two backends by setting `JUDGE_BACKEND` below — no out-of-band CLI state, everything is configured in this cell and recorded in the saved artifact:
  - **`"openai"`** — a hosted OpenAI judge. Needs `OPENAI_API_KEY` in the environment; `JUDGE_MODEL` names the judge (e.g. `gpt-4o-mini`).
  - **`"local"`** — any OpenAI-compatible endpoint (local NIM/vLLM/TGI, or another cloud). Set `JUDGE_MODEL`, `JUDGE_BASE_URL`, and (if needed) `JUDGE_API_KEY`. Nothing leaves your network.

  The next cell turns whichever backend you pick into a single judge object that all the metrics share, so switching judges is a one-line change here.

> **Judge and model-under-test are fully independent** — different backends, endpoints, and models. You can evaluate a local vLLM model with an OpenAI judge, or keep everything on-prem. Use a judge that is **stronger than (and different from) the model under test** — a model grading its own homework inflates scores.

> **Local-judge JSON errors.** DeepEval asks the judge for strict JSON and parses it; weak judges wrap it in prose or ```` ```json ```` fences, giving `ValueError: Evaluation LLM outputted an invalid JSON`. The local judge below defends against this by requesting **structured output** and returning a validated object (deepeval then skips its own text parsing), with a lenient JSON-recovery fallback. If you still hit it, the real fix is a **more capable judge** — a 7B+ instruct model handles the JSON contract reliably; sub-1B models often don't. `evaluate(..., ignore_errors=True)` (Section 5) keeps one bad case from aborting the whole run.

In [ ]:
import json
import os
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)

# ---- Model under test (any OpenAI-compatible endpoint) -------------------
MODEL = ""        # model name the endpoint serves, e.g. "Qwen/Qwen3-0.6B"
BASE_URL = ""     # e.g. "http://localhost:8000/v1"  (note the /v1)
API_KEY = os.environ.get("UNDER_TEST_API_KEY", "none")  # most local endpoints ignore it
TEMPERATURE = 0.0  # deterministic-ish outputs -> more reproducible scores
MAX_TOKENS = 300

# ---- Judge model (used by the DeepEval metrics) --------------------------
# Pick the backend, then fill in the fields it uses. The next cell builds a
# single judge object from these — nothing depends on `deepeval` CLI state.
JUDGE_BACKEND = "openai"           # "openai" (hosted) or "local" (any OpenAI-compatible endpoint)

JUDGE_MODEL = "gpt-4o-mini"        # judge model name (both backends)

# Only used when JUDGE_BACKEND == "local":
JUDGE_BASE_URL = "http://localhost:8001/v1"                   # judge endpoint (note the /v1)
JUDGE_API_KEY = os.environ.get("JUDGE_API_KEY", "none")       # most local endpoints ignore it

assert MODEL and BASE_URL, "Set MODEL and BASE_URL above."
assert JUDGE_BACKEND in ("openai", "local"), "JUDGE_BACKEND must be 'openai' or 'local'."
if JUDGE_BACKEND == "openai" and not os.environ.get("OPENAI_API_KEY"):
    print("WARNING: JUDGE_BACKEND='openai' but OPENAI_API_KEY is not set — "
          "either export it, or switch JUDGE_BACKEND to 'local'.")

print(f"Model under test: {MODEL} @ {BASE_URL}")
if JUDGE_BACKEND == "openai":
    print(f"Judge: {JUDGE_MODEL} (OpenAI)")
else:
    print(f"Judge: {JUDGE_MODEL} @ {JUDGE_BASE_URL} (local / OpenAI-compatible)")

In [ ]:
# Build ONE judge object from the config above; every metric shares it.
#
# - "openai": pass the model-name string straight through — deepeval's default
#   path, uses OPENAI_API_KEY.
# - "local":  wrap any OpenAI-compatible endpoint in a tiny DeepEvalBaseLLM.
#   We talk to it with the same `openai` client the rest of the repo uses, so
#   there's no dependency on `deepeval set-local-model` / CLI global state, and
#   the judge endpoint is captured in-notebook (and in the saved artifact).
#
# Why the `schema` handling below matters: deepeval metrics ask the judge for
# strict JSON, then parse the raw string. Smaller/local judges often wrap it in
# prose or ```json fences, which makes deepeval raise
#   "Evaluation LLM outputted an invalid JSON. Please use a better model."
# deepeval avoids that entirely if generate(prompt, schema=Model) returns a
# Pydantic instance — it then trusts the object instead of parsing text. We do
# that via the OpenAI structured-output path, with a lenient fallback for
# endpoints that don't support it.

if JUDGE_BACKEND == "openai":
    judge = JUDGE_MODEL  # metric(model="gpt-4o-mini") — deepeval's built-in OpenAI path
else:
    import json as _json
    import re as _re
    from openai import OpenAI
    from deepeval.models.base_model import DeepEvalBaseLLM

    def _coerce_json(text: str) -> dict:
        """Best-effort recovery of a JSON object from a chatty judge reply."""
        text = text.strip()
        # strip ```json ... ``` / ``` ... ``` fences
        fence = _re.match(r"^```(?:json)?\s*(.*?)\s*```$", text, _re.DOTALL)
        if fence:
            text = fence.group(1).strip()
        try:
            return _json.loads(text)
        except _json.JSONDecodeError:
            # fall back to the first {...} span in the reply
            start, end = text.find("{"), text.rfind("}")
            if start != -1 and end > start:
                return _json.loads(text[start:end + 1])
            raise

    class OpenAICompatibleJudge(DeepEvalBaseLLM):
        """Judge backed by any OpenAI-compatible /v1/chat/completions endpoint.

        Honors deepeval's optional `schema` (a Pydantic model): when present we
        request structured output and return a model instance, so deepeval never
        has to parse the raw string — this is what fixes the "invalid JSON" error
        for weaker local judges.
        """

        def __init__(self, model: str, base_url: str, api_key: str):
            self.model = model
            self._client = OpenAI(base_url=base_url, api_key=api_key)

        def load_model(self):
            return self._client

        def _complete(self, prompt: str, schema=None):
            messages = [{"role": "user", "content": prompt}]
            if schema is None:
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=0.0,
                )
                return (resp.choices[0].message.content or "").strip()

            # --- schema requested: try native structured output first ----------
            try:
                parsed = self._client.beta.chat.completions.parse(
                    model=self.model, messages=messages, temperature=0.0,
                    response_format=schema,
                ).choices[0].message.parsed
                if parsed is not None:
                    return parsed  # already a validated Pydantic instance
            except Exception:
                pass  # endpoint may not support .parse / response_format=Model

            # --- fallback: ask for JSON, then coerce + validate ourselves ------
            try:
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=0.0,
                    response_format={"type": "json_object"},
                )
            except Exception:
                # last resort: no json mode at all, nudge via the prompt
                messages = [{"role": "user",
                             "content": prompt + "\n\nRespond with ONLY a valid JSON object."}]
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=0.0,
                )
            raw = (resp.choices[0].message.content or "").strip()
            return schema.model_validate(_coerce_json(raw))

        def generate(self, prompt: str, schema=None, *args, **kwargs):
            return self._complete(prompt, schema)

        async def a_generate(self, prompt: str, schema=None, *args, **kwargs):
            # deepeval uses the async path when running metrics concurrently;
            # the OpenAI client call is blocking, so just delegate.
            return self._complete(prompt, schema)

        def get_model_name(self) -> str:
            return self.model

    judge = OpenAICompatibleJudge(JUDGE_MODEL, JUDGE_BASE_URL, JUDGE_API_KEY)

print(f"Judge object ready: {judge if isinstance(judge, str) else judge.get_model_name()}")

## 2. Golden dataset

A small, curated set of `(input, expected_output)` pairs. Same design philosophy as this repo's prompt files (see the UC9 notebook): each record is a **self-contained test case representative of one real task**, and the set is small enough that every case earns its place.

`expected_output` is a *reference answer*, not a string to match verbatim — the GEval judge compares substance (facts, coverage, intent), not wording. The tasks below deliberately mirror the repo's v1 **Content Generation** scope.

In [ ]:
eval_dataset = [
    {
        "input": "What is the capital of France? Answer in one sentence.",
        "expected_output": "The capital of France is Paris.",
    },
    {
        "input": "List three practical benefits of database connection pooling.",
        "expected_output": (
            "1. Avoids the overhead of opening a new connection per request, reducing latency. "
            "2. Reuses a bounded set of connections, protecting the database from being "
            "overwhelmed by too many simultaneous connections. "
            "3. Improves scalability and throughput under concurrent load."
        ),
    },
    {
        "input": (
            "Draft a short, friendly email to a customer explaining that their reported "
            "bug has been fixed in today's release and how to update."
        ),
        "expected_output": (
            "A brief, friendly email that: thanks the customer for the bug report, "
            "states the bug is fixed in today's release, and tells them how to get the "
            "fix (update to the latest version)."
        ),
    },
    {
        "input": (
            "Summarize in two sentences: Monolithic architectures are simpler to deploy "
            "and iterate on for small teams, but scale poorly organizationally. "
            "Microservices allow independent scaling and deployment per service, at the "
            "cost of significant operational complexity (service discovery, distributed "
            "tracing, network failure handling) that small teams usually cannot afford."
        ),
        "expected_output": (
            "Monoliths are simpler to deploy and iterate on, making them a good fit for "
            "small teams despite organizational scaling limits. Microservices enable "
            "independent scaling and deployment but add operational complexity that "
            "small teams typically cannot afford."
        ),
    },
]

print(f"{len(eval_dataset)} golden test cases:")
for i, c in enumerate(eval_dataset):
    print(f"  [{i}] {c['input'][:70]}...")

## 3. Generate outputs from the model under test

One chat completion per test case, `temperature=0` for repeatability. This is the only cell that talks to the model under test — everything after it is judging.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def generate(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return (resp.choices[0].message.content or "").strip()

for i, case in enumerate(eval_dataset):
    case["actual_output"] = generate(case["input"])
    print(f"[{i}] {case['input'][:50]}...")
    print(f"    -> {case['actual_output'][:100]}...\n")

## 4. Build test cases & metrics

Each `LLMTestCase` bundles `input` + `actual_output` (+ `expected_output` for reference-based metrics). Two metrics, both LLM-as-judge, both returning a 0–1 score with a written reason:

| Metric | What it checks | Needs `expected_output` | Threshold |
|---|---|---|---|
| **Correctness** (GEval, custom criteria) | actual output is substantively consistent with the reference answer | yes | 0.5 |
| **Answer Relevancy** (built-in) | output actually addresses the input, no off-topic filler | no | 0.7 |

A case **passes** a metric when its score ≥ the metric's threshold. Both metrics use the shared `judge` object built in Section 1.

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval, AnswerRelevancyMetric

# The evaluation-params enum was renamed in newer deepeval releases
# (LLMTestCaseParams -> SingleTurnParams); support both.
try:
    from deepeval.test_case import LLMTestCaseParams as Params
except ImportError:
    from deepeval.test_case import SingleTurnParams as Params

test_cases = [
    LLMTestCase(
        input=c["input"],
        actual_output=c["actual_output"],
        expected_output=c["expected_output"],
    )
    for c in eval_dataset
]

# `judge` (built in Section 1) is either an OpenAI model-name string or a
# DeepEvalBaseLLM instance — metric(model=...) accepts both. Passing the object
# (not JUDGE_MODEL) is what keeps a local judge from falling back to OpenAI.
correctness = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual output is factually and substantively "
        "consistent with the expected output. Penalize missing key facts, "
        "contradictions, and unmet instructions from the input; do NOT penalize "
        "differences in wording, formatting, or reasonable extra detail."
    ),
    evaluation_params=[Params.INPUT, Params.ACTUAL_OUTPUT, Params.EXPECTED_OUTPUT],
    threshold=0.5,
    model=judge,
)
relevancy = AnswerRelevancyMetric(threshold=0.7, model=judge)

print(f"{len(test_cases)} test cases, 2 metrics (Correctness, Answer Relevancy)")

## 5. Run the evaluation

`evaluate()` runs every metric against every test case (each score = one or more judge calls, so expect ~a few seconds per case) and prints a per-case report.

In [ ]:
from deepeval import evaluate

# ignore_errors=True: if the judge still fails a single case (e.g. an
# unrecoverable non-JSON reply on a weak local model), skip that case instead
# of aborting the whole run — you still get partial results to inspect. Drop it
# once the judge is solid if you want failures to be loud.
try:
    results = evaluate(
        test_cases=test_cases,
        metrics=[correctness, relevancy],
        ignore_errors=True,
    )
except TypeError:
    # older deepeval: evaluate() took a config object / different kwargs
    results = evaluate(test_cases=test_cases, metrics=[correctness, relevancy])

## 6. Results

Flatten into one row per (test case, metric): score, pass/fail, and the judge's reason — the reason is the debugging payload; read it before trusting or disputing a score. The table is also written to `artifacts/` as JSON so the run can be committed alongside performance runs (repo convention: reproducibility = Git).

In [ ]:
import pandas as pd

test_results = getattr(results, "test_results", results)  # older deepeval returned a plain list

rows = []
for i, tr in enumerate(test_results):
    for md in tr.metrics_data or []:
        rows.append({
            "case": i,
            "input": eval_dataset[i]["input"][:60] + "...",
            "metric": md.name,
            "score": round(md.score, 3) if md.score is not None else None,
            "passed": md.success,
            "reason": md.reason,
        })

df = pd.DataFrame(rows)

summary = df.groupby("metric").agg(
    mean_score=("score", "mean"),
    pass_rate=("passed", "mean"),
    cases=("passed", "size"),
).round(3)
print(f"Model under test: {MODEL}")

# Aggregate pass rate / mean score per metric (Correctness, Answer Relevancy):
display(summary)

# Per-case detail: one row per (test case, metric) with score, pass/fail, and
# the judge's written reason.
print("Per-case results:")
with pd.option_context("display.max_colwidth", 120):
    display(df)

In [ ]:
# Persist the run so it can be committed (mirrors how AIPerf artifacts are kept).
out_dir = REPO_ROOT / "notebooks" / "artifacts" / "deepeval-model-eval"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"results_{MODEL.replace('/', '_')}.json"

# Record the exact judge used so the run is reproducible / auditable.
judge_record = {"backend": JUDGE_BACKEND, "model": JUDGE_MODEL}
if JUDGE_BACKEND == "local":
    judge_record["base_url"] = JUDGE_BASE_URL

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": MODEL,
            "base_url": BASE_URL,
            "judge": judge_record,
            "temperature": TEMPERATURE,
            "cases": eval_dataset,
            "results": rows,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print(f"Saved -> {out_path}")

### Notes

- **Comparing models** = rerun Sections 3–6 with a different `MODEL`/`BASE_URL` and diff the saved JSONs — same pattern as the performance suites, where the workload stays fixed and only the endpoint changes. Combined with the Model Selection suite's TTFT/ITL/goodput numbers, this gives both axes of the model-selection decision (fast *and* correct). **Keep the judge fixed** while comparing models — the judge is recorded in each artifact, so a mismatched judge is easy to spot.
- **Switching judges** is a one-line change in Section 1: `JUDGE_BACKEND = "openai"` vs `"local"` (plus the endpoint fields for local). The `OpenAICompatibleJudge` wrapper talks to any OpenAI-compatible `/v1` endpoint with the same `openai` client the rest of the repo uses — no `deepeval` CLI global state, so the judge config lives in the notebook and in the saved artifact. Note the metrics take `model=judge` (the object), **not** `model=JUDGE_MODEL` — passing the string name is what makes deepeval fall back to its default OpenAI judge and demand `OPENAI_API_KEY`.
- **LLM-as-judge caveats**: scores are judge-dependent — pin the judge per run (it's recorded in the saved JSON), keep `temperature=0` on the model under test, and spot-check reasons rather than trusting scores blindly. Small score deltas (< ~0.1) between models are noise. A local judge should still be **stronger than and different from** the model under test.
- **Growing this**: the workflow scales by adding rows to `eval_dataset` (or loading real prompts from `model-selection/prompts/content_generation.jsonl` and writing reference answers for them), and by adding metrics — DeepEval ships `FaithfulnessMetric`/`HallucinationMetric` (needs `retrieval_context`, relevant when the RAG scenario lands in v2), `BiasMetric`, `ToxicityMetric`, and fully custom `GEval` criteria.
- **Out of scope here, by design**: no Confident AI cloud login (`deepeval login`) — everything runs and is stored locally, matching this repo's raw-artifact convention.